# Bachelor AI Thesis: Retrieval Playground & Metrics Viewer

This interactive notebook allows you to play with the RAG systems, run custom queries, view generated note summaries, and inspect the final thesis evaluation metrics.

### System Configurations:
1. **Baseline RAG (Flat Chunks)**: Retrieves fixed-size character chunks directly from raw PDF texts using SentenceTransformer (`all-MiniLM-L6-v2`) and cosine similarity.
2. **Structured RAG (Generated Notes)**: Retrieves full agent-generated notes (YAML frontmatter + text summaries).
3. **Link-Expanded RAG (Graph Re-ranked)**: Seeded by Structured RAG, parses inter-document `[[WikiLinks]]` mapping them to arXiv IDs, applies connectivity score boost ($\alpha = 0.05$), and re-ranks the corpus.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path to enable imports of the src package
repo_root = Path("__file__").resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import json
import numpy as np
import pandas as pd
from IPython.display import display, HTML, Markdown

# Import retrieval and evaluation functions
from src.retrieval.baseline import retrieve as baseline_retrieve, load_index as load_baseline_index
from src.retrieval.structured import retrieve as structured_retrieve
from src.retrieval.link_expanded import retrieve_link_expanded
from src.evaluation.retrieval_eval import evaluate_query

# Constants and paths
queries_path = Path("data/queries/queries.json")
baseline_index_path = Path("data/results/baseline_index.json")
structured_index_path = Path("data/results/structured_index.json")
notes_dir = Path("data/generated_notes")
metrics_path = Path("data/results/retrieval_metrics.json")

def load_paper_titles(notes_dir):
    titles = {}
    import re
    for note_path in notes_dir.glob("*.md"):
        doc_id = note_path.stem
        try:
            content = note_path.read_text(encoding="utf-8", errors="ignore")
            title_match = re.search(r'^title:\s*"(.*?)"', content, re.MULTILINE)
            if not title_match:
                title_match = re.search(r'^title:\s*(.*?)$', content, re.MULTILINE)
            if title_match:
                titles[doc_id] = title_match.group(1).strip().strip('"').strip("'")
            else:
                titles[doc_id] = doc_id
        except Exception:
            titles[doc_id] = doc_id
    return titles

def retrieve_structured_with_scores(query, index_path, top_k=10):
    with open(index_path, encoding="utf-8") as f:
        index_data = json.load(f)
    doc_ids = index_data["doc_ids"]
    doc_embs = np.array(index_data["embeddings"])
    model_name = index_data.get("model", "all-MiniLM-L6-v2")
    
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
    
    model = SentenceTransformer(model_name)
    query_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    similarities = cosine_similarity(query_emb, doc_embs)[0]
    
    results = []
    for i in range(len(doc_ids)):
        results.append((float(similarities[i]), doc_ids[i]))
    results = sorted(results, key=lambda x: x[0], reverse=True)
    return results[:top_k]

def retrieve_link_expanded_with_scores(query, index_path, notes_dir, top_k=10, graph_boost_alpha=0.05):
    with open(index_path, encoding="utf-8") as f:
        index_data = json.load(f)
    doc_ids = index_data["doc_ids"]
    doc_embs = np.array(index_data["embeddings"])
    model_name = index_data.get("model", "all-MiniLM-L6-v2")
    
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
    
    model = SentenceTransformer(model_name)
    query_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    similarities = cosine_similarity(query_emb, doc_embs)[0]
    
    doc_scores = {doc_ids[i]: float(similarities[i]) for i in range(len(doc_ids))}
    initial_results = sorted(doc_ids, key=lambda x: doc_scores[x], reverse=True)[:top_k]
    
    from src.retrieval.link_expanded import build_title_map, get_linked_notes
    title_map = build_title_map(notes_dir)
    
    linked_docs = set()
    for doc_id in initial_results:
        note_path = notes_dir / f"{doc_id}.md"
        linked_targets = get_linked_notes(note_path)
        for target in linked_targets:
            matched_id = title_map.get(target.lower())
            if matched_id and matched_id in doc_scores and matched_id not in initial_results:
                linked_docs.add(matched_id)
                
    boosted_scores = dict(doc_scores)
    for l_doc in linked_docs:
        boosted_scores[l_doc] += graph_boost_alpha
        
    re_ranked = sorted(doc_ids, key=lambda x: boosted_scores[x], reverse=True)[:top_k]
    return [(boosted_scores[doc], doc, doc in linked_docs) for doc in re_ranked]

print("[OK] Packages, models, and custom retrieval wrappers loaded successfully.")

In [ ]:
# Load the indices
print("Loading Baseline Index...")
docs, embeddings, model = load_baseline_index(str(baseline_index_path))

print("Loading Structured Index...")
with open(structured_index_path, encoding="utf-8") as f:
    struct_index = json.load(f)

# Load list of queries
with open(queries_path, encoding="utf-8") as f:
    queries = json.load(f)

print(f"[OK] Indices and {len(queries)} evaluation queries loaded successfully.")

## 1. Interactive Retrieval Sandbox

Type a custom query below and execute the cell. It will retrieve the top documents from all three systems and display them side-by-side.

In [ ]:
# ==================== PLAYGROUND INPUTS ====================
# Feel free to change the query and top_k below!
user_query = "What is RAPTOR and how does it organize retrieval?"
top_k = 5
# ==========================================================

# 1. Run Baseline
baseline_chunks = baseline_retrieve(user_query, docs, embeddings, model, k=top_k, mode="baseline")
baseline_list = []
for score, chunk in baseline_chunks:
    baseline_list.append(f"<b>Doc: {chunk['doc_id']}</b> (Score: {score:.4f})<br>{chunk['text'][:200]}...")

# 2. Run Structured
structured_results = retrieve_structured_with_scores(user_query, structured_index_path, top_k=top_k)
structured_list = []
titles_map = load_paper_titles(notes_dir)
for score, doc_id in structured_results:
    title = titles_map.get(doc_id, doc_id)
    summary = ""
    note_path = notes_dir / f"{doc_id}.md"
    if note_path.exists():
        text_content = note_path.read_text(encoding="utf-8", errors="ignore")
        parts = text_content.split("## Summary")
        if len(parts) > 1:
            summary = parts[1].split("##")[0].strip()[:180]
        else:
            summary = text_content.split("---")[-1].strip()[:180]
    structured_list.append(f"<b>Doc: {doc_id}</b> (Score: {score:.4f})<br><b>Title:</b> {title}<br><i>Summary excerpt:</i> {summary}...")

# 3. Run Link-Expanded
link_expanded_results = retrieve_link_expanded_with_scores(user_query, structured_index_path, notes_dir, top_k=top_k, graph_boost_alpha=0.05)
link_expanded_list = []
for score, doc_id, is_boosted in link_expanded_results:
    title = titles_map.get(doc_id, doc_id)
    summary = ""
    note_path = notes_dir / f"{doc_id}.md"
    if note_path.exists():
        text_content = note_path.read_text(encoding="utf-8", errors="ignore")
        parts = text_content.split("## Summary")
        if len(parts) > 1:
            summary = parts[1].split("##")[0].strip()[:180]
        else:
            summary = text_content.split("---")[-1].strip()[:180]
    boost_label = " <span style='color: green; font-weight: bold;'>[Boosted +0.05]</span>" if is_boosted else ""
    link_expanded_list.append(f"<b>Doc: {doc_id}</b> (Score: {score:.4f}){boost_label}<br><b>Title:</b> {title}<br><i>Summary excerpt:</i> {summary}...")

# RENDER SIDE-BY-SIDE IN HTML
html_content = f"""
<h3>Query: <i>\"{user_query}\"</i></h3>
<table style=\"width:100%; border: 1px solid #ddd; border-collapse: collapse;\">
  <tr style=\"background-color: #f2f2f2; text-align: left;\">
    <th style=\"padding: 10px; border: 1px solid #ddd; width: 33%;\">Baseline RAG (Flat Chunks)</th>
    <th style=\"padding: 10px; border: 1px solid #ddd; width: 33%;\">Structured RAG (Notes)</th>
    <th style=\"padding: 10px; border: 1px solid #ddd; width: 33%;\">Link-Expanded RAG (Graph Re-ranked)</th>
  </tr>
"""

for idx in range(top_k):
    b_val = baseline_list[idx] if idx < len(baseline_list) else "N/A"
    s_val = structured_list[idx] if idx < len(structured_list) else "N/A"
    l_val = link_expanded_list[idx] if idx < len(link_expanded_list) else "N/A"
    
    # Highlight differences
    l_style = ""
    if idx < len(link_expanded_results) and idx < len(structured_results) and link_expanded_results[idx][1] != structured_results[idx][1]:
        l_style = "background-color: #e6f7ff;" # highlight re-ranked nodes
        
    html_content += f"""
  <tr>
    <td style=\"padding: 10px; border: 1px solid #ddd; vertical-align: top; font-size: 13px;\">{b_val}</td>
    <td style=\"padding: 10px; border: 1px solid #ddd; vertical-align: top; font-size: 13px;\">{s_val}</td>
    <td style=\"padding: 10px; border: 1px solid #ddd; vertical-align: top; font-size: 13px; {l_style}\">{l_val}</td>
  </tr>
  """

html_content += "</table>"
display(HTML(html_content))

## 2. Generated Obsidian Note Explorer

Enter an arXiv ID to view the full generated markdown note (extracted metadata YAML, summaries, and links) rendered nicely in the notebook.

In [ ]:
# ==================== EXPLORER INPUT ====================
# Enter one of your corpus arXiv IDs (e.g. 2401.18059, 2005.11401, 2505.13138, etc.)
arxiv_id_to_view = "2401.18059"
# ========================================================

note_path = notes_dir / f"{arxiv_id_to_view}.md"
if note_path.exists():
    text_content = note_path.read_text(encoding="utf-8")
    parts = text_content.split("---")
    if len(parts) >= 3:
        frontmatter = parts[1].strip()
        body = "---".join(parts[2:]).strip()
        
        # Parse YAML frontmatter lines
        lines = [line.strip() for line in frontmatter.split("\n") if line.strip()]
        html_front = """
        <div style=\"background-color: #f7f9fa; border-left: 5px solid #007acc; padding: 15px; border-radius: 4px; margin-bottom: 20px; font-family: sans-serif;\">
          <h4 style=\"margin-top: 0; color: #007acc;\">Paper Metadata</h4>
          <table style=\"width: 100%; border-collapse: collapse;\">
        """
        for line in lines:
            if ":" in line:
                key, val = line.split(":", 1)
                key = key.strip().replace("_", " ").title()
                val = val.strip().strip('"').strip("'")
                if key.lower() in ["relatedpapers", "links to"] and (val == "[]" or not val):
                    continue
                html_front += f"<tr><td style='font-weight: bold; width: 150px; padding: 4px 0;'>{key}:</td><td>{val}</td></tr>"
        
        import re
        body_links = re.findall(r"\[\[([^\]]+)\]\]", body)
        seen_links = []
        for l in body_links:
            clean_l = l.split("|")[0].strip()
            if clean_l not in seen_links:
                seen_links.append(clean_l)
        
        if seen_links:
            html_front += f"<tr><td style='font-weight: bold; width: 150px; padding: 4px 0; vertical-align: top;'>Graph Connections:</td><td>"
            html_front += ", ".join([f"<code>[[{l}]]</code>" for l in seen_links])
            html_front += "</td></tr>"
            
        html_front += "</table></div>"
        
        display(HTML(html_front))
        display(Markdown(body))
    else:
        display(Markdown(text_content))
else:
    print(f"Error: Note for ID '{arxiv_id_to_view}' not found. Available IDs: {[p.stem for p in notes_dir.glob('*.md')]}")

## 3. Evaluation Results Dashboard

Below is the comparative metrics table computed over all 40 queries against the gold standard labels.

In [ ]:
# Load gold labels
gold_labels_path = Path("data/gold_labels/gold_labels.json")
if not gold_labels_path.exists():
    print(f"Error: {gold_labels_path} not found.")
else:
    with open(gold_labels_path, encoding="utf-8") as f:
        gold_data = json.load(f)
    
    print("Running evaluations live across all 40 queries...")
    baseline_metrics_list = []
    structured_metrics_list = []
    link_expanded_metrics_list = []
    
    for q in queries:
        qid = q["query_id"]
        text = q["text"]
        gold_relevant = gold_data.get(qid, [])
        
        # 1. Baseline
        retrieved_chunks = baseline_retrieve(text, docs, embeddings, model, k=50, mode="baseline")
        seen_docs = []
        for score, chunk in retrieved_chunks:
            doc_id = chunk["doc_id"]
            if doc_id not in seen_docs:
                seen_docs.append(doc_id)
        b_res = seen_docs[:10]
        b_met = evaluate_query(b_res, gold_relevant, k_values=[1, 3, 5, 10])
        baseline_metrics_list.append(b_met)
        
        # 2. Structured
        s_res = structured_retrieve(text, structured_index_path, top_k=10)
        s_met = evaluate_query(s_res, gold_relevant, k_values=[1, 3, 5, 10])
        structured_metrics_list.append(s_met)
        
        # 3. Link-Expanded
        l_res = retrieve_link_expanded(text, structured_index_path, notes_dir, top_k=10, graph_boost_alpha=0.05)
        l_met = evaluate_query(l_res, gold_relevant, k_values=[1, 3, 5, 10])
        link_expanded_metrics_list.append(l_met)
    
    def average_metrics(metrics_list):
        avg = {}
        keys = metrics_list[0].keys()
        for k in keys:
            avg[k] = np.mean([m[k] for m in metrics_list])
        return avg
    
    baseline_avg = average_metrics(baseline_metrics_list)
    structured_avg = average_metrics(structured_metrics_list)
    link_expanded_avg = average_metrics(link_expanded_metrics_list)
    
    # Render the dynamic table
    metrics_md = f"""
| Metric | Baseline RAG | Structured RAG | Link-Expanded RAG | Description |
|---|---|---|---|---|
| **Recall@1** | {baseline_avg.get('recall@1', 0.0):.4f} | {structured_avg.get('recall@1', 0.0):.4f} | {link_expanded_avg.get('recall@1', 0.0):.4f} | Retrieval matches target in top slot |
| **Precision@1** | {baseline_avg.get('precision@1', 0.0):.4f} | {structured_avg.get('precision@1', 0.0):.4f} | {link_expanded_avg.get('precision@1', 0.0):.4f} | Proportion of top slot matches that are relevant |
| **F1@1** | {baseline_avg.get('f1@1', 0.0):.4f} | {structured_avg.get('f1@1', 0.0):.4f} | {link_expanded_avg.get('f1@1', 0.0):.4f} | Harmonic mean of Precision and Recall at top slot |
| **Recall@3** | {baseline_avg.get('recall@3', 0.0):.4f} | {structured_avg.get('recall@3', 0.0):.4f} | {link_expanded_avg.get('recall@3', 0.0):.4f} | Retrieval matches target in top-3 slots |
| **Recall@5** | {baseline_avg.get('recall@5', 0.0):.4f} | {structured_avg.get('recall@5', 0.0):.4f} | {link_expanded_avg.get('recall@5', 0.0):.4f} | Retrieval matches target in top-5 slots |
| **Recall@10** | {baseline_avg.get('recall@10', 0.0):.4f} | {structured_avg.get('recall@10', 0.0):.4f} | {link_expanded_avg.get('recall@10', 0.0):.4f} | Retrieval matches target in top-10 slots |
| **Precision@10** | {baseline_avg.get('precision@10', 0.0):.4f} | {structured_avg.get('precision@10', 0.0):.4f} | {link_expanded_avg.get('precision@10', 0.0):.4f} | Match ratio over 10 slots |
| **F1@10** | {baseline_avg.get('f1@10', 0.0):.4f} | {structured_avg.get('f1@10', 0.0):.4f} | {link_expanded_avg.get('f1@10', 0.0):.4f} | Harmonic mean at 10 slots |
| **MRR** | {baseline_avg.get('mrr', 0.0):.4f} | {structured_avg.get('mrr', 0.0):.4f} | {link_expanded_avg.get('mrr', 0.0):.4f} | Mean Reciprocal Rank (ranking quality) |
"""
    display(Markdown(metrics_md))